In [1]:
# install the neede libraries
%pip install pyiqa
%pip install pandas
%pip install scikit-learn
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# import the needed libraries
import os
import numpy as np
import pandas as pd
import torch
import pyiqa
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# paths
ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"
QUALITY_THRESHOLD = 0.20

In [4]:
# loading MUSIQ for quality testing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
musiq_metric = pyiqa.create_metric("musiq", device=device)
print("MUSIQ Loaded")

Device: cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [5]:
# loading detection scores
det_df = pd.read_csv("face_detection_scores.csv")

print("Detection Scores Loaded")
print(det_df.shape)

Detection Scores Loaded
(3168, 6)


In [6]:
# quality score function
def get_quality_score(image_path):
    score = musiq_metric(image_path)
    return float(score.cpu().item())

In [19]:
def get_detection_score(split, identity, image_name):

    # remove degradation suffixes
    base_name = image_name

    degradation_tags = [
        "_blur_low",
        "_blur_high",
        "_jpeg_low",
        "_jpeg_high",
    ]

    for tag in degradation_tags:
        base_name = base_name.replace(tag, "")

    row = det_df[
        (det_df["split"] == split)
        & (det_df["subset"] == "probe")
        & (det_df["identity"] == identity)
        & (det_df["image_name"] == base_name)
    ]

    if len(row) == 0:
        return np.nan

    return float(row.iloc[0]["det_score"])

In [20]:
sample_identity = os.listdir(
    os.path.join(ROOT_IMAGES, "train", "degraded_probes")
)[0]

sample_file = os.listdir(
    os.path.join(ROOT_IMAGES, "train", "degraded_probes", sample_identity)
)[0]

print(sample_file)

Salma_Hayek_0005_blur_low.jpg


In [21]:
# load gallery embeddings
def load_gallery_db(gallery_root):
    gallery_db = {}
    total = 0

    for identity in os.listdir(gallery_root):
        identity_dir = os.path.join(gallery_root, identity)
        gallery_db[identity] = []

        for file in os.listdir(identity_dir):
            emb = np.load(os.path.join(identity_dir, file))
            gallery_db[identity].append(emb)
            total += 1

    print("Loaded Gallery Embeddings:", total)
    return gallery_db

In [22]:
# cosine similarity verification
def search_gallery(probe_embedding, gallery_db):
    identity_scores = {}

    for identity in gallery_db:
        sims = []

        for gallery_emb in gallery_db[identity]:
            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]
            sims.append(sim)

        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)
    predicted_identity = sorted_scores[0][0]
    best_similarity = sorted_scores[0][1]
    second_similarity = sorted_scores[1][1]
    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [23]:
# generating dataset for training our model
def generate_dataset(split):

    gallery_root = os.path.join(ROOT_EMBEDDINGS, split, "gallery")
    probe_emb_root = os.path.join(ROOT_EMBEDDINGS, split, "degraded_probes")
    probe_img_root = os.path.join(ROOT_IMAGES, split, "degraded_probes")
    gallery_db = load_gallery_db(gallery_root)
    full_rows = []

    accepted = 0
    rejected = 0
    missing_images = 0

    counter = 0

    for identity in tqdm(os.listdir(probe_emb_root)):
        emb_dir = os.path.join(probe_emb_root, identity)
        img_dir = os.path.join(probe_img_root, identity)

        for emb_file in os.listdir(emb_dir):
            emb_path = os.path.join(emb_dir, emb_file)
            probe_embedding = np.load(emb_path)
            base_name = os.path.splitext(emb_file)[0]
            image_path = None

            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = os.path.join(img_dir, base_name + ext)

                if os.path.exists(candidate):
                    image_path = candidate
                    break

            if image_path is None:
                missing_images += 1
                continue

            quality_score = get_quality_score(image_path)
            image_name = os.path.basename(image_path)
            det_score = get_detection_score(
                split,
                identity,
                image_name,
            )

            if quality_score < QUALITY_THRESHOLD:
                rejected += 1
                continue

            accepted += 1

            predicted_identity, best_similarity, second_similarity, margin = (
                search_gallery(probe_embedding, gallery_db)
            )

            label = int(predicted_identity == identity)

            full_rows.append(
                {
                    "quality_score": quality_score,
                    "det_score": det_score,
                    "best_similarity": best_similarity,
                    "second_best_similarity": second_similarity,
                    "margin": margin,
                    "true_identity": identity,
                    "predicted_identity": predicted_identity,
                    "label": label,
                }
            )

            counter += 1

            if counter % 100 == 0:
                print(f"\nProcessed: {counter}")
                print(f"Quality: {quality_score:.4f}")
                print(f"Best Similarity: {best_similarity:.4f}")
                print(f"Second Similarity: {second_similarity:.4f}")
                print(f"Margin: {margin:.4f}")
                print(f"Prediction: {predicted_identity}")
                print(f"Truth: {identity}")
                print(f"Label: {label}")

    full_df = pd.DataFrame(full_rows)
    svm_df = full_df[["quality_score", "det_score", "best_similarity", "margin", "label"]]
    full_csv = os.path.join(ROOT_EMBEDDINGS, f"{split}_features_full.csv")
    svm_csv = os.path.join(ROOT_EMBEDDINGS, f"{split}_svm.csv")
    full_df.to_csv(full_csv, index=False)
    svm_df.to_csv(svm_csv, index=False)

    print(f"\nAccepted: {accepted}")
    print(f"Rejected: {rejected}")
    print(f"Missing Images: {missing_images}")
    print(f"\nSaved: {full_csv}")
    print(f"Saved: {svm_csv}")

    return full_df

In [ ]:
# generating dataset for training our model
train_df = generate_dataset("train")

Loaded Gallery Embeddings: 2002


  9%|▉         | 9/100 [00:40<06:43,  4.43s/it]


Processed: 100
Quality: 23.0040
Best Similarity: 0.6450
Second Similarity: 0.2532
Margin: 0.3918
Prediction: Bill_McBride
Truth: Bill_McBride
Label: 1


 15%|█▌        | 15/100 [01:19<07:28,  5.28s/it]


Processed: 200
Quality: 20.5297
Best Similarity: 0.6408
Second Similarity: 0.3047
Margin: 0.3361
Prediction: Taha_Yassin_Ramadan
Truth: Taha_Yassin_Ramadan
Label: 1


...

 76%|███████▌  | 76/100 [10:36<02:23,  5.97s/it]


Processed: 1600
Quality: 13.4655
Best Similarity: 0.5547
Second Similarity: 0.3898
Margin: 0.1650
Prediction: Hugo_Chavez
Truth: Hugo_Chavez
Label: 1


 78%|███████▊  | 78/100 [11:15<04:30, 12.28s/it]


Processed: 1700
Quality: 19.1161
Best Similarity: 0.6793
Second Similarity: 0.3055
Margin: 0.3738
Prediction: Halle_Berry
Truth: Halle_Berry
Label: 1


 82%|████████▏ | 82/100 [11:57<03:33, 11.88s/it]

In [12]:
#check to see the columns in dataset
print(train_df.columns)

print(train_df["det_score"].describe())

print(
    train_df.groupby("label")["det_score"].describe()
)

Index(['quality_score', 'det_score', 'best_similarity',
       'second_best_similarity', 'margin', 'true_identity',
       'predicted_identity', 'label'],
      dtype='object')
count    2196.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: det_score, dtype: float64
        count  mean  std  min  25%  50%  75%  max
label                                            
0        41.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0
1      2155.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0


In [13]:
# stats
print("\nTRAIN DATASET")
print(train_df.shape)
print("\nLabel Counts")
print(train_df["label"].value_counts())
print("\nQuality Statistics")
print(train_df["quality_score"].describe())
print("\nMargin Statistics")
print(train_df["margin"].describe())


TRAIN DATASET
(2196, 8)

Label Counts
label
1    2155
0      41
Name: count, dtype: int64

Quality Statistics
count    2196.000000
mean       20.093970
std         5.794668
min        12.476462
25%        15.741508
50%        18.516455
75%        22.908837
max        45.461479
Name: quality_score, dtype: float64

Margin Statistics
count    2196.000000
mean        0.371866
std         0.123941
min         0.000351
25%         0.303187
50%         0.388449
75%         0.457939
max         0.669552
Name: margin, dtype: float64


In [16]:
print(det_df["subset"].unique())

['gallery' 'probe']
